# Complete Ingestion Pipeline for Erica AI Tutor

This notebook combines all ingestion tasks:
1. **Connection Testing** - Verify MongoDB and Ollama
2. **Webpage Scraping** - Crawl course website
3. **Slide Extraction** - Extract text from linked PDFs and PPTX files
4. **Verification** - Show all ingested content and statistics

## Installation Requirements

Run this cell first to install required packages:

In [1]:
!pip install pymongo requests beautifulsoup4 PyPDF2 python-pptx lxml youtube-transcript-api



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Import Libraries

In [2]:
import sys
sys.path.append('/workspace')

from pymongo import MongoClient
from datetime import datetime
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
import re
from typing import Set, List, Dict, Any
from collections import Counter

# PDF and PPTX extraction
import PyPDF2
from pptx import Presentation
import io

from ingestion.mongo_helper import MongoHelper

## 1. MongoDB Helper Class

Handles all database operations.

In [3]:
## Implemented in mongo_helper.py

## 2. Test Connections

Verify MongoDB and Ollama are accessible.

In [4]:
def test_mongodb():
    """Test MongoDB connection"""
    print("Testing MongoDB connection...")
    try:
        mongo = MongoHelper()
        counts = mongo.count_documents()
        print("MongoDB connected successfully!")
        print(f"  Current counts: {counts}")
        return mongo, True
    except Exception as e:
        print(f"MongoDB connection failed: {e}")
        return None, False

def test_ollama():
    """Test Ollama connection"""
    print("\nTesting Ollama connection...")
    try:
        response = requests.get("http://ollama:11434/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json()
            print("Ollama connected successfully!")
            print(f"  Available models: {[m['name'] for m in models.get('models', [])]}")
            return True
        else:
            print(f"Ollama returned status code: {response.status_code}")
            return False
    except Exception as e:
        print(f"Ollama connection failed: {e}")
        return False

# Run connection tests
mongo, mongodb_ok = test_mongodb()
ollama_ok = test_ollama()

if mongodb_ok and ollama_ok:
    print("\n" + "="*70)
    print("All systems ready!")
    print("="*70)
else:
    print("\n" + "="*70)
    print("Some systems failed - check configuration")
    print("="*70)

Testing MongoDB connection...
MongoDB connected successfully!
  Current counts: {'webpages': 131, 'slides': 19, 'videos': 53}

Testing Ollama connection...
Ollama connected successfully!
  Available models: ['qwen2.5:7b']

All systems ready!


## 3. Slide Extraction Functions

Extract text from PDF and PPTX files.

In [5]:
def extract_pdf_text(pdf_content: bytes) -> str:
    """Extract text from PDF bytes"""
    try:
        pdf_file = io.BytesIO(pdf_content)
        pdf_reader = PyPDF2.PdfReader(pdf_file)
        
        text_content = []
        for page in pdf_reader.pages:
            text_content.append(page.extract_text())
        
        return '\n'.join(text_content)
    except Exception as e:
        print(f"  Error extracting PDF text: {e}")
        return None

def extract_pptx_text(pptx_content: bytes) -> str:
    """Extract text from PPTX bytes"""
    try:
        pptx_file = io.BytesIO(pptx_content)
        presentation = Presentation(pptx_file)
        
        text_content = []
        for slide in presentation.slides:
            slide_text = []
            for shape in slide.shapes:
                if hasattr(shape, "text"):
                    slide_text.append(shape.text)
            text_content.append('\n'.join(slide_text))
        
        return '\n\n'.join(text_content)
    except Exception as e:
        print(f"  Error extracting PPTX text: {e}")
        return None

# Tags whose href attribute points at build assets or metadata, never at
# navigable content -- used when parsing full HTML pages (the .md fallback
# path below doesn't have a <head>, so this only matters for HTML mode).
NON_CONTENT_TAGS = {'link', 'script', 'meta', 'style'}

def find_slide_links_in_page(soup: BeautifulSoup, base_url: str) -> List[str]:
    """Find all PDF and PPTX links in an HTML page (HTML fallback path)."""
    slide_urls = set()
    for tag in soup.find_all(href=True):
        if tag.name in NON_CONTENT_TAGS:
            continue
        full_url = urljoin(base_url, tag['href'])
        if full_url.lower().endswith(('.pdf', '.pptx', '.ppt')):
            slide_urls.add(full_url)
    return list(slide_urls)

# --- Markdown/MDX helpers (primary path) -----------------------------------
# Mintlify serves a raw MDX/markdown version of every doc page at
# "<url>.md" -- no nav chrome, no build assets, headers/structure intact.
# It still contains JSX-like components (<Card href="...">, <Frame>, etc.)
# and standard markdown links ([text](url)), so link discovery here uses
# regex instead of BeautifulSoup's tag search.
HREF_ATTR_RE = re.compile(r'href\s*=\s*["\']([^"\']+)["\']')
MD_LINK_RE = re.compile(r'\[[^\]]*\]\(([^)\s]+)(?:\s+"[^"]*")?\)')
MD_TITLE_RE = re.compile(r'^#\s+(.+)$', re.MULTILINE)
# Boilerplate repeated verbatim on every page of this site -- strip it so it
# doesn't get chunked/embedded 40+ times over as if it were course content.
MD_DOC_INDEX_BANNER_RE = re.compile(r'\A(?:>.*\n?)+')
MD_EDIT_GITHUB_LINE_RE = re.compile(r'^.*Edit this page on GitHub.*$\n?', re.MULTILINE)

def find_links_in_markdown(text: str, base_url: str) -> List[str]:
    """Find all http(s) links in raw markdown/MDX text via regex (both
    JSX-style href="..." attributes and standard [text](url) links)."""
    urls = set()
    for m in HREF_ATTR_RE.finditer(text):
        urls.add(urljoin(base_url, m.group(1)))
    for m in MD_LINK_RE.finditer(text):
        urls.add(urljoin(base_url, m.group(1)))
    return list(urls)

def find_slide_links_in_markdown(text: str, base_url: str) -> List[str]:
    """Find PDF/PPTX links in raw markdown/MDX text."""
    return [u for u in find_links_in_markdown(text, base_url)
            if u.lower().endswith(('.pdf', '.pptx', '.ppt'))]

def clean_markdown_content(text: str) -> str:
    """Strip the repeated 'Documentation Index' banner and GitHub-edit
    boilerplate, then strip remaining JSX-like tags (<Frame>, <Card>,
    <Callout>, <img>, etc.) while keeping their inner text -- reusing
    BeautifulSoup as a lightweight tag-stripper, not an HTML parser."""
    text = MD_DOC_INDEX_BANNER_RE.sub('', text, count=1)
    text = MD_EDIT_GITHUB_LINE_RE.sub('', text)
    text = BeautifulSoup(text, 'html.parser').get_text(separator='\n', strip=True)
    return text

def extract_markdown_title(text: str, fallback_url: str) -> str:
    m = MD_TITLE_RE.search(text)
    if m:
        return m.group(1).strip()
    return fallback_url


# --- YouTube video detection and transcript extraction -----------------
# Erica spec calls for video ingestion with automatic transcription and
# timecode mapping, sourced from course "media" pages (e.g.
# /media/robotics/development-environment) which embed YouTube links via
# <Card href="https://www.youtube.com/watch?v=..."> components -- the same
# href pattern already used for slide discovery, just checked against a
# YouTube URL instead of a .pdf/.pptx extension.

YOUTUBE_URL_RE = re.compile(r'(?:youtube\.com/watch\?v=|youtu\.be/)([A-Za-z0-9_-]{11})')

def extract_youtube_id(url: str):
    m = YOUTUBE_URL_RE.search(url)
    return m.group(1) if m else None

def find_video_links_in_page(soup: BeautifulSoup, base_url: str) -> List[str]:
    """Find YouTube links in an HTML page (HTML fallback path)."""
    video_urls = set()
    for tag in soup.find_all(href=True):
        if tag.name in NON_CONTENT_TAGS:
            continue
        full_url = urljoin(base_url, tag['href'])
        if extract_youtube_id(full_url):
            video_urls.add(full_url)
    return list(video_urls)

def find_video_links_in_markdown(text: str, base_url: str) -> List[str]:
    """Find YouTube links in raw markdown/MDX text."""
    return [u for u in find_links_in_markdown(text, base_url) if extract_youtube_id(u)]

def fetch_youtube_transcript(video_id: str):
    """Fetch timecoded transcript segments for a YouTube video. Returns a
    list of {'text', 'start', 'duration'} dicts, or None if captions are
    genuinely unavailable (auto-generated captions are fine; many lecture
    videos won't have manually-authored ones).

    youtube-transcript-api v1.0+ replaced the old static
    YouTubeTranscriptApi.get_transcript(video_id) with an instance method,
    api.fetch(video_id), returning a FetchedTranscript object of
    FetchedTranscriptSnippet dataclasses (.text/.start/.duration attributes,
    not dict keys) instead of a plain list of dicts. .to_raw_data()
    converts it back to that same list-of-dicts shape so nothing downstream
    (insert_video, M3's video processing) needs to change.

    IpBlocked/RequestBlocked (YouTube rate-limiting the container's IP) are
    RE-RAISED rather than swallowed -- these are transient and worth
    stopping the whole batch for, not worth retrying per-video. Everything
    else (TranscriptsDisabled, NoTranscriptFound, VideoUnavailable, etc.)
    is genuinely permanent for that video and is swallowed, returning None,
    same as before."""
    from youtube_transcript_api import YouTubeTranscriptApi
    from youtube_transcript_api._errors import IpBlocked, RequestBlocked

    try:
        api = YouTubeTranscriptApi()
        fetched = api.fetch(video_id)
        return fetched.to_raw_data()
    except (IpBlocked, RequestBlocked):
        raise
    except Exception as e:
        print(f"    Transcript fetch failed for {video_id}: {type(e).__name__}: {e}")
        return None


## 4. Complete Ingestion Pipeline

Main class that orchestrates webpage scraping and slide extraction.

In [6]:
class CompleteIngestion:
    def __init__(self, mongo: MongoHelper):
        self.mongo = mongo
        self.visited_urls: Set[str] = set()
        self.slide_files: Set[str] = set()
        self.base_url = "https://aegean.ai"
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/124.0.0.0 Safari/537.36"
        }
        # /media was previously excluded as "marketing" content. It is
        # NOT marketing -- it hosts video lecture pages (e.g.
        # /media/robotics/development-environment) with embedded YouTube
        # links, which is the video ingestion source the Erica spec
        # requires. /products, /blog, /about remain excluded.
        self.allowed_prefixes = ('/book', '/courses', '/aiml-common', '/media')
        self.video_files: Set[str] = set()
        # Track which fetch path was used per page, for visibility into how
        # much of the crawl actually got the clean markdown vs HTML fallback.
        self.fetch_stats = {'markdown': 0, 'html': 0, 'failed': 0}

    @staticmethod
    def normalize_url(url: str) -> str:
        """Collapse equivalent URL forms down to one canonical form so the
        same page is never crawled/stored twice under different spellings:
        - trailing slash:  '/book/introduction/'   -> '/book/introduction'
        - literal .md link: '/foo/index.md'         -> '/foo/index' -> '/foo'
        - folder index:    '/foo/index'             -> '/foo'
        (Mintlify serves identical content at '/foo' and '/foo/index' --
        the folder-index convention common to most static site generators.)
        """
        parsed = urlparse(url)
        path = parsed.path
        if path.lower().endswith('.md'):
            path = path[:-3]
        while True:
            if len(path) > 1 and path.endswith('/'):
                path = path[:-1]
                continue
            if path.lower().endswith('/index'):
                path = path[:-len('/index')]
                continue
            break
        if path == '':
            path = '/'
        return parsed._replace(path=path, fragment='').geturl()

    def is_valid_url(self, url: str) -> bool:
        """Check if URL is valid, within course content, and is real
        content (not a build asset, sitemap, or marketing page)."""
        parsed = urlparse(url)

        if 'aegean.ai' not in parsed.netloc:
            return False

        if parsed.path not in ('', '/') and not parsed.path.startswith(self.allowed_prefixes):
            return False

        if '/mintlify-assets/' in parsed.path:
            return False

        skip_filenames = ['sitemap.xml', 'robots.txt', 'llms.txt', 'llms-full.txt',
                           'manifest.json', 'favicon.ico']
        if parsed.path.rsplit('/', 1)[-1] in skip_filenames:
            return False

        skip_extensions = [
            '.pdf', '.pptx', '.ppt', '.zip', '.jpg', '.jpeg', '.png', '.gif',
            '.svg', '.webp', '.mp4', '.css', '.js', '.woff', '.woff2', '.ttf',
            '.eot', '.ico', '.json', '.xml', '.map', '.yaml', '.yml'
        ]
        path_only = parsed.path.lower()
        if any(path_only.endswith(ext) for ext in skip_extensions):
            return False

        return True

    def fetch_content(self, url: str):
        """Fetch a page, preferring the clean markdown source.

        Mintlify serves raw MDX/markdown at "<url>.md" for doc pages -- no
        nav chrome, no build assets, structure intact. This tries that
        first and falls back to the normal HTML page if the .md route
        404s, isn't supported for this particular path, or errors for any
        reason. Per-URL fallback means the pipeline works correctly
        regardless of whether every route on the site actually supports
        the .md convention -- nothing needs to be verified up front.

        Returns (raw_text, is_markdown: bool) or (None, None) on failure.
        """
        parsed = urlparse(url)
        if parsed.path not in ('', '/'):
            md_url = url.rstrip('/') + '.md'
            try:
                resp = requests.get(md_url, timeout=15, headers=self.headers)
                if resp.status_code == 200 and resp.text.strip():
                    self.fetch_stats['markdown'] += 1
                    return resp.text, True
            except requests.RequestException:
                pass

        try:
            resp = requests.get(url, timeout=15, headers=self.headers)
            resp.raise_for_status()
            self.fetch_stats['html'] += 1
            return resp.text, False
        except requests.RequestException as e:
            self.fetch_stats['failed'] += 1
            raise

    def ingest_webpage(self, url: str, max_depth: int = 3, current_depth: int = 0):
        """Recursively fetch and store webpage content, plus extract slides.
        Handles both the markdown and HTML fetch paths."""
        url = self.normalize_url(url)
        if url in self.visited_urls or current_depth > max_depth:
            return

        self.visited_urls.add(url)

        try:
            print(f"{'  ' * current_depth}Fetching: {url}")
            raw_text, is_markdown = self.fetch_content(url)

            if is_markdown:
                title = extract_markdown_title(raw_text, url)
                slide_urls = find_slide_links_in_markdown(raw_text, url)
                video_urls = find_video_links_in_markdown(raw_text, url)
                links = [u for u in find_links_in_markdown(raw_text, url)
                         if self.is_valid_url(u) and self.normalize_url(u) not in self.visited_urls]
                content = clean_markdown_content(raw_text)
            else:
                soup = BeautifulSoup(raw_text, 'html.parser')
                title = soup.title.string if soup.title else url
                slide_urls = find_slide_links_in_page(soup, url)
                video_urls = find_video_links_in_page(soup, url)
                for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
                    tag.decompose()
                content = soup.get_text(separator='\n', strip=True)
                links = self.extract_links_html(soup, url)

            for slide_url in slide_urls:
                if slide_url not in self.slide_files:
                    self.slide_files.add(slide_url)
                    print(f"{'  ' * current_depth}  Found slide: {slide_url.split('/')[-1]}")

            for video_url in video_urls:
                if video_url not in self.video_files:
                    self.video_files.add(video_url)
                    print(f"{'  ' * current_depth}  Found video: {video_url}")

            metadata = {
                'depth': current_depth,
                'word_count': len(content.split()),
                'slides_found': len(slide_urls),
                'source_format': 'markdown' if is_markdown else 'html',
            }

            self.mongo.insert_webpage(url, title, content, metadata)
            print(f"{'  ' * current_depth}Stored webpage ({'md' if is_markdown else 'html'}): {str(title)[:50]}...")

            if current_depth < max_depth:
                time.sleep(0.3)  # Be nice to the server
                links = [self.normalize_url(u) for u in links]
                links = list(dict.fromkeys(links))  # dedupe, preserve order
                for link in links:
                    self.ingest_webpage(link, max_depth, current_depth + 1)

        except Exception as e:
            print(f"{'  ' * current_depth}Failed: {url} - {e}")

    def extract_links_html(self, soup: BeautifulSoup, base_url: str) -> List[str]:
        """Link discovery for the HTML fallback path. Searches href on any
        tag (not just <a>) since Mintlify renders internal navigation via
        components like <Card href="...">, excluding tags that only ever
        point at build assets."""
        links = []
        for tag in soup.find_all(href=True):
            if tag.name in NON_CONTENT_TAGS:
                continue
            url = urljoin(base_url, tag['href'])
            url = url.split('#')[0]
            if self.is_valid_url(url):
                links.append(url)
        return links

    def ingest_slides(self):
        """Download and extract text from all discovered slide files"""
        print("\n" + "="*70)
        print(f"EXTRACTING SLIDE CONTENT ({len(self.slide_files)} files)")
        print("="*70)

        if not self.slide_files:
            print("No slide files found.")
            return

        success_count = 0
        for slide_url in self.slide_files:
            filename = slide_url.split('/')[-1]
            print(f"\nProcessing: {filename}")

            try:
                response = requests.get(slide_url, timeout=30, headers=self.headers)
                response.raise_for_status()

                if slide_url.lower().endswith('.pdf'):
                    content = extract_pdf_text(response.content)
                elif slide_url.lower().endswith(('.pptx', '.ppt')):
                    content = extract_pptx_text(response.content)
                else:
                    print(f"  Unsupported file type")
                    continue

                if content:
                    metadata = {
                        'word_count': len(content.split()),
                        'file_type': slide_url.split('.')[-1].lower()
                    }

                    self.mongo.insert_slide(
                        url=slide_url,
                        filename=filename,
                        content=content,
                        metadata=metadata
                    )
                    print(f"  Content extracted ({metadata['word_count']} words)")
                    success_count += 1
                else:
                    print(f"  Could not extract content")

            except Exception as e:
                print(f"  Failed: {e}")

        print(f"\nSuccessfully extracted {success_count}/{len(self.slide_files)} slide files")

    def ingest_videos(self):
        """Fetch transcripts (with timecodes) for all discovered YouTube
        videos and store them as 'video' resources, matching the Erica
        spec's Resource schema: type in [pdf, slide, video, web], with
        timecodes for videos alongside the span used for pdf/slide pages.

        Resumable: videos already in mongo.db.videos are skipped, so a
        rerun after an IP block cools down doesn't waste requests re-
        fetching ones that already succeeded -- only the actual gap gets
        retried. A ~1.5s delay between requests reduces the chance of
        tripping YouTube's rate limiter in the first place. If IpBlocked/
        RequestBlocked happens anyway, the whole batch stops immediately
        rather than burning through the remaining videos on guaranteed
        failures -- there's no point hammering a blocked IP 30 more times."""
        from youtube_transcript_api._errors import IpBlocked, RequestBlocked

        print("\n" + "="*70)
        print(f"EXTRACTING VIDEO TRANSCRIPTS ({len(self.video_files)} videos)")
        print("="*70)

        if not self.video_files:
            print("No videos found.")
            return

        already_have = {doc['url'] for doc in self.mongo.db.videos.find({}, {'url': 1})}
        remaining = [v for v in self.video_files if v not in already_have]
        if already_have:
            print(f"Skipping {len(self.video_files) - len(remaining)} already-ingested videos "
                  f"(resuming from a previous run)")

        success_count = 0
        blocked = False

        for video_url in remaining:
            video_id = extract_youtube_id(video_url)
            if not video_id:
                print(f"  Skipping (couldn't parse video ID): {video_url}")
                continue

            print(f"\nProcessing: {video_url} (id={video_id})")
            try:
                transcript = fetch_youtube_transcript(video_id)
                if not transcript:
                    print(f"  No transcript available (captions may be disabled)")
                    continue

                full_text = ' '.join(seg['text'] for seg in transcript)
                metadata = {
                    'word_count': len(full_text.split()),
                    'segment_count': len(transcript),
                    'duration_seconds': transcript[-1]['start'] + transcript[-1].get('duration', 0) if transcript else 0,
                }

                self.mongo.insert_video(
                    url=video_url,
                    video_id=video_id,
                    content=full_text,
                    segments=transcript,
                    metadata=metadata
                )
                print(f"  Transcript extracted ({metadata['word_count']} words, {metadata['segment_count']} segments)")
                success_count += 1
                time.sleep(1.5)  # space out requests to avoid tripping rate limits

            except (IpBlocked, RequestBlocked) as e:
                print(f"\n  BLOCKED by YouTube ({type(e).__name__}) -- stopping video ingestion here.")
                print(f"  {success_count} succeeded this run before the block hit.")
                print(f"  Wait a while (minutes to hours, YouTube doesn't document exact cooldowns) "
                      f"and re-run -- already-ingested videos will be skipped automatically.")
                blocked = True
                break

            except Exception as e:
                print(f"  Failed: {e}")

        total_have = len(already_have) + success_count
        print(f"\n{'Stopped early due to IP block. ' if blocked else ''}"
              f"{total_have}/{len(self.video_files)} video transcripts ingested total "
              f"({success_count} this run).")

    def run_complete_ingestion(self, start_urls: List[str], max_depth: int = 3):
        """Run the complete ingestion pipeline"""
        print("="*70)
        print("STARTING COMPLETE INGESTION PIPELINE")
        print("="*70)

        print("\n[STEP 1/2] Scraping webpages...")
        for url in start_urls:
            self.ingest_webpage(url, max_depth=max_depth)

        print(f"\nScraped {len(self.visited_urls)} webpages")
        print(f"  via markdown: {self.fetch_stats['markdown']}, via HTML fallback: {self.fetch_stats['html']}, failed: {self.fetch_stats['failed']}")
        print(f"Found {len(self.slide_files)} slide files")

        print("\n[STEP 2/3] Extracting slide content...")
        self.ingest_slides()

        print("\n[STEP 3/3] Extracting video transcripts...")
        self.ingest_videos()

        print("\n" + "="*70)
        print("INGESTION COMPLETE!")
        print("="*70)


## 5. Run the Complete Ingestion

Set CLEAR_EXISTING_DATA = True to start fresh, or False to append to existing data.

In [7]:
# Configuration
CLEAR_EXISTING_DATA = True  # Clears webpages + slides before ingestion (cheap/fast to redo)
CLEAR_VIDEOS = False        # Videos are rate-limited and slow to (re-)fetch -- keep False
                             # unless you specifically want to wipe and re-fetch all video
                             # transcripts. Left False, ingest_videos() below skips videos
                             # already in Mongo and only fetches what's missing -- this is
                             # what makes re-running after an IpBlocked cooldown cheap.

if CLEAR_EXISTING_DATA:
    print("Clearing webpages and slides...")
    mongo.db.webpages.delete_many({})
    mongo.db.slides.delete_many({})
    print()

if CLEAR_VIDEOS:
    print("Clearing videos...")
    mongo.db.videos.delete_many({})
    print()

ingestion = CompleteIngestion(mongo)

# Seed with stable hub pages rather than every individual chapter path.
# The site's chapter slugs have been renamed twice during this project
# (dnn -> training-dnns, kinematics+state-estimation merged, etc.), so
# hardcoding a full chapter list is a maintenance trap. book/introduction
# and the course pages link out to every chapter/lecture via <Card> links,
# which extract_links_html / find_links_in_markdown now both discover --
# so recursive crawl from these few entry points reaches the same content
# without needing to track the site's internal reorganizations by hand.
start_urls = [
    "https://aegean.ai/",
    "https://aegean.ai/book/introduction",
    "https://aegean.ai/courses/",
    "https://aegean.ai/courses/ai/",
    "https://aegean.ai/courses/cv/",

    # Not reachable via recursive discovery from the seeds above --
    # book/introduction's own chapter list doesn't mention VAEs/generative
    # models at all, so nothing links to this content from anywhere the
    # crawler otherwise reaches. Confirmed via a Mongo content search for
    # "jensen" returning zero hits, then tracked down directly. This is
    # exactly the Jensen's Inequality / ELBO / variational lower bound
    # content the M4 test question needs and previously had nothing to
    # retrieve for.
    "https://aegean.ai/aiml-common/lectures/vae/vae-architecture/index",
    "https://aegean.ai/aiml-common/lectures/vae/elbo-optimization",
    "https://aegean.ai/aiml-common/lectures/diffusion/introduction/index",
]

# max_depth=5: hub page -> chapter overview -> lecture index -> lecture
# subpage is a 3-4 hop chain on this site; the allowed_prefixes scoping
# keeps this from fanning out into unrelated content even at this depth.
ingestion.run_complete_ingestion(start_urls, max_depth=5)


Clearing webpages and slides...

STARTING COMPLETE INGESTION PIPELINE

[STEP 1/2] Scraping webpages...
Fetching: https://aegean.ai/
Stored webpage (html): Home - aegean.ai...
  Fetching: https://aegean.ai/book/introduction
  Stored webpage (md): Introduction...
    Fetching: https://aegean.ai/book/logic
    Stored webpage (md): Logic...
      Fetching: https://aegean.ai/aiml-common/lectures/logical-reasoning/logical-agents
      Stored webpage (md): Logical Agents...
      Fetching: https://aegean.ai/aiml-common/lectures/logical-reasoning/propositional-logic
      Stored webpage (md): Propositional Logic...
      Fetching: https://aegean.ai/aiml-common/lectures/logical-reasoning/automated-reasoning
      Stored webpage (md): Automated Reasoning...
        Fetching: https://aegean.ai/aiml-common/lectures/logical-reasoning/applications
        Stored webpage (md): Applications of Automated Reasoning...
        Fetching: https://aegean.ai/aiml-common/lectures/logical-reasoning
        Sto

/tmp/ipykernel_1782/3705268196.py:87: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  text = BeautifulSoup(text, 'html.parser').get_text(separator='\n', strip=True)


          Stored webpage (md): LLM-driven world and model authoring...
      Fetching: https://aegean.ai/aiml-common/lectures/scene-understanding/object-detection/faster-rcnn
      Stored webpage (md): Faster RCNN...
      Fetching: https://aegean.ai/aiml-common/lectures/scene-understanding/object-detection/faster-rcnn/pytorch/02_backbone/02_backbone
      Stored webpage (md): Backbone, ResNet50 and FPN...
        Fetching: https://aegean.ai/aiml-common/lectures/scene-understanding/object-detection/faster-rcnn/pytorch/02_backbone/02_backbone-slides
        Stored webpage (md): Backbone — ResNet50 and FPN — Slides...
      Fetching: https://aegean.ai/aiml-common/lectures/scene-understanding/scene-understanding-intro
        Found slide: P18-1238.pdf
      Stored webpage (md): Introduction...
    Fetching: https://aegean.ai/book/physical-ai
    Stored webpage (md): Physical AI...
      Fetching: https://aegean.ai/aiml-common/lectures/sim2real
      Stored webpage (md): Sim-to-Real Transf

## 6. Verification & Statistics

Display comprehensive statistics about ingested content.

In [8]:
# Get document counts
counts = mongo.count_documents()

print("="*70)
print("INGESTION SUMMARY")
print("="*70)
print(f"Webpages:      {counts['webpages']:3d}")
print(f"Slides:        {counts['slides']:3d}")
print(f"TOTAL:         {sum(counts.values()):3d}")

# Calculate total words
total_words = 0
word_counts = []

# Webpage words
for doc in mongo.db.webpages.find({}, {'metadata.word_count': 1}):
    wc = doc.get('metadata', {}).get('word_count', 0)
    total_words += wc
    word_counts.append(wc)

# Slide words
for doc in mongo.db.slides.find({}, {'metadata.word_count': 1}):
    wc = doc.get('metadata', {}).get('word_count', 0)
    total_words += wc

print("\n" + "="*70)
print("CONTENT STATISTICS")
print("="*70)
print(f"Total words ingested:       {total_words:,}")
if counts['webpages'] > 0:
    print(f"Average words per webpage:  {total_words // max(sum(counts.values()), 1):,}")
    print(f"Min words per webpage:      {min(word_counts) if word_counts else 0:,}")
    print(f"Max words per webpage:      {max(word_counts) if word_counts else 0:,}")

INGESTION SUMMARY
Webpages:      144
Slides:         22
TOTAL:         219

CONTENT STATISTICS
Total words ingested:       689,312
Average words per webpage:  3,147
Min words per webpage:      32
Max words per webpage:      6,596


## 7. List All Ingested URLs

Required for M2 milestone - show all URLs that were ingested.

In [9]:
print("="*70)
print("ALL INGESTED URLs (M2 REQUIREMENT)")
print("="*70)

for doc_type, url in mongo.get_all_urls():
    print(f"[{doc_type:8s}] {url}")

ALL INGESTED URLs (M2 REQUIREMENT)
[webpage ] https://aegean.ai/
[webpage ] https://aegean.ai/book/introduction
[webpage ] https://aegean.ai/book/logic
[webpage ] https://aegean.ai/aiml-common/lectures/logical-reasoning/logical-agents
[webpage ] https://aegean.ai/aiml-common/lectures/logical-reasoning/propositional-logic
[webpage ] https://aegean.ai/aiml-common/lectures/logical-reasoning/automated-reasoning
[webpage ] https://aegean.ai/aiml-common/lectures/logical-reasoning/applications
[webpage ] https://aegean.ai/aiml-common/lectures/logical-reasoning
[webpage ] https://aegean.ai/aiml-common/lectures/logical-reasoning/logical-inference
[webpage ] https://aegean.ai/book/llms
[webpage ] https://aegean.ai/aiml-common/lectures/nlp/nlp-introduction/nlp-pipelines
[webpage ] https://aegean.ai/aiml-common/lectures/rnn/lstm
[webpage ] https://aegean.ai/aiml-common/lectures/nlp/transformers/transformers-intro
[webpage ] https://aegean.ai/aiml-common/lectures/nlp/nmt/nmt-intro
[webpage ] https:

## 8. Sample Content from Each Type

In [10]:
print("="*70)
print("SAMPLE WEBPAGE")
print("="*70)
webpage = mongo.db.webpages.find_one()
if webpage:
    print(f"Title: {webpage['title']}")
    print(f"URL: {webpage['url']}")
    print(f"Word Count: {webpage['metadata'].get('word_count', 'N/A')}")
    print(f"\nFirst 300 chars:\n{webpage['content'][:300]}...\n")

print("="*70)
print("SAMPLE SLIDE CONTENT")
print("="*70)
slide = mongo.db.slides.find_one()
if slide:
    print(f"Filename: {slide['filename']}")
    print(f"URL: {slide.get('url', 'N/A')}")
    print(f"Word Count: {slide['metadata'].get('word_count', 'N/A')}")
    print(f"\nFirst 300 chars:\n{slide['content'][:300]}...\n")
else:
    print("No slides found.\n")

SAMPLE WEBPAGE
Title: Home - aegean.ai
URL: https://aegean.ai/
Word Count: 304

First 300 chars:
Home - aegean.ai
Documentation Index
Fetch the complete documentation index at:
/llms.txt
Use this file to discover all available pages before exploring further.
Skip to main content
Aegean AI
Build agents that see, reason, and act
From perception to language to physical manipulation, we research an...

SAMPLE SLIDE CONTENT
Filename: 1704.04503.pdf
URL: https://arxiv.org/pdf/1704.04503.pdf
Word Count: 7227

First 300 chars:
Improving Object Detection With One Line of Code
Navaneeth Bodla* Bharat Singh* Rama Chellappa Larry S. Davis
Center For Automation Research, University of Maryland, College Park
fnbodla,bharat,rama,lsd g@umiacs.umd.edu
Abstract
Non-maximum suppression is an integral part of the ob-
ject detection p...



## 9. Topic Distribution Analysis

In [11]:
# Extract topics from URLs
topics = []
for doc_type, url in mongo.get_all_urls():
    parts = url.split('/')
    for part in parts:
        if part and part not in [
            'https:', '',
            'aegean.ai',
            'www.youtube.com', 'youtu.be',   # video domain -- every video shares these,
                                              # so like aegean.ai they're pure noise, not signal
            'aiml-common', 'lectures',
            'book', 'courses', 'media',
            'index',
        ]:
            topics.append(part.replace('-', ' ').replace('_', ' '))

topic_counts = Counter(topics)

print("="*70)
print("TOP 20 TOPICS IN INGESTED CONTENT")
print("="*70)

for topic, count in topic_counts.most_common(20):
    print(f"{count:3d} - {topic}")

TOP 20 TOPICS IN INGESTED CONTENT
 28 - scene understanding
 19 - object detection
 15 - pytorch
 10 - arxiv.org
 10 - pdf
  9 - semantic segmentation
  8 - faster rcnn
  8 - planning
  7 - logical reasoning
  7 - maskrcnn
  7 - task planning
  7 - diffusion
  6 - yolo
  6 - rse
  5 - nlp
  5 - optimization
  5 - 02 backbone
  5 - ml math
  4 - introduction
  4 - mdp


## 10. Export Summary for M2 Submission

Generate a text file with all URLs for easy submission.

In [12]:
with open('/workspace/M2_ingested_urls.txt', 'w') as f:
    f.write("="*70 + "\n")
    f.write("M2 MILESTONE - ALL INGESTED URLs\n")
    f.write("="*70 + "\n\n")
    
    counts = mongo.count_documents()
    f.write(f"Total Webpages: {counts['webpages']}\n")
    f.write(f"Total Slides:   {counts['slides']}\n")
    f.write(f"TOTAL:          {sum(counts.values())}\n\n")
    
    f.write("="*70 + "\n")
    f.write("ALL URLS\n")
    f.write("="*70 + "\n\n")
    
    for doc_type, url in mongo.get_all_urls():
        f.write(f"[{doc_type:8s}] {url}\n")

print("Exported URL list to: /workspace/M2_ingested_urls.txt")

Exported URL list to: /workspace/M2_ingested_urls.txt


## Summary

This notebook provides a complete ingestion pipeline that:

1. Scrapes course webpages recursively
2. Discovers linked PDF and PPTX files
3. Extracts text content from slides
4. Stores everything in MongoDB with proper metadata
5. Provides comprehensive statistics and verification

Next Steps:
- Move on to M3: Knowledge Graph Construction
- Use the ingested content to extract concepts and build the pedagogical graph

In [15]:
## DEBUG
for doc in mongo.db.webpages.find():
    content_lower = doc['content'].lower()
    if 'elbo' in content_lower or 'evidence lower bound' in content_lower or 'variational lower bound' in content_lower:
        print(f"Title: {doc['title']}")
        print(f"URL: {doc['url']}")
        idx = content_lower.find('elbo')
        if idx == -1:
            idx = content_lower.find('lower bound')
        print(f"Context: {doc['content'][idx:idx+500]}\n")

Title: VAE architecture
URL: https://aegean.ai/aiml-common/lectures/vae/vae-architecture
Context: ELBO)** — a tractable surrogate for the marginal log-likelihood that is derived from the KL divergence between $q$ and the true posterior. The derivation and its consequences for joint optimization are covered in the [Optimization and the ELBO](/aiml-common/lectures/vae/elbo-optimization/index) page.

## PyTorch reference

| PyTorch class                                                                    | Description                                                                   |
| --------

Title: Optimization and the ELBO
URL: https://aegean.ai/aiml-common/lectures/vae/elbo-optimization
Context: ELBO

> Derivation of the Evidence Lower Bound from the KL divergence between the inference model and the true posterior, and why maximizing the ELBO jointly trains the encoder and decoder.

The [VAE architecture](/aiml-common/lectures/vae/vae-architecture/index) introduces an inference netw